# 02. Unit Economics: CAC, LTV, Payback

Phase 3, 2026-08-21. Answers Question 1 (blended unit economics) and the unit-economics half of Question 2 (segment divergence). Question numbers refer to `docs/locked_business_questions.md`.

**Two CAC methods are used deliberately, for different questions, and neither is silently swapped in for the other:**

- **Lag-adjusted trailing window** (`compute_cac`): spend is shifted earlier by the segment's average sales cycle before being divided by wins in the target period, since spend in month M produces wins several months later, not in the same month. Used for the "current" blended CAC headline. Not used for the Enterprise pre-push era, because the required spend history would reach back before `sm_spend`'s first row (`WINDOW_START`), which does not exist, and would silently understate the founder-era figure.
- **Same-period era CAC** (`compute_era_cac`): total spend during an era divided by total wins closing during that era, no lag shift. Used for the pre-push vs. post-push Enterprise comparison, where the question is "how efficiently did spend convert to wins during this era," not "what did this specific cohort of wins cost."

Both are shown with the window they use stated explicitly.

In [1]:
import sys
sys.path.insert(0, "../src")
import phase3_lib as lib
import pandas as pd
pd.set_option("display.width", 160)

tables = lib.load_all()
deals, customers, spend = tables["deals"], tables["customers"], tables["sm_spend"]

## Question 1: blended unit economics

### CAC, trailing 12 months (lag-adjusted), blended and by segment

In [2]:
period_start = lib.WINDOW_END - pd.Timedelta(days=365)
cac_rows = []
for seg in lib.SEGMENTS:
    r = lib.compute_cac(deals, spend, seg, period_start, lib.WINDOW_END)
    cac_rows.append(r)
cac_df = pd.DataFrame(cac_rows)[["segment", "lag_months", "spend_usd", "wins", "cac"]]

# Blended: sum spend and wins across segments over the same trailing window,
# each segment lag-adjusted by its own cycle length before summing.
blended_spend = sum(r["spend_usd"] for r in cac_rows)
blended_wins = sum(r["wins"] for r in cac_rows)
blended_cac = blended_spend / blended_wins
print(f"Blended CAC, trailing 12mo: ${blended_cac:,.0f}  (spend ${blended_spend:,.0f} / {blended_wins} wins)")
cac_df

Blended CAC, trailing 12mo: $22,963  (spend $6,200,000 / 270 wins)


,segment,lag_months,spend_usd,wins,cac
0,Enterprise,4.19,3200000.04,29,110344.828966
1,Mid-Market,1.95,1800000.00,101,17821.782178
2,SMB,1.13,1200000.00,140,8571.428571


### LTV, both methods, and payback

Primary method: `ACV x 0.78 gross margin x min(1/annual_churn, 3 years)`. Naive contrast: same formula with the cap removed. Annual churn is **measured empirically** for Mid-Market and SMB, which have enough observed churn events (24 and 100) to support it, and taken from the **locked benchmark** (8%, Optifai, 5-10% sensitivity range) for Enterprise, which does not.

In [3]:
ltv_rows = []
for seg in lib.SEGMENTS:
    acv = deals[(deals["segment"] == seg) & (deals["deal_won"])]["deal_size_usd"].mean()
    if seg == "Enterprise":
        churn = lib.ENTERPRISE_BENCHMARK_CHURN
        churn_source = "benchmark (Optifai), Enterprise retention unmeasurable from this data"
        events, exposure = None, None
    else:
        churn, events, exposure = lib.empirical_annual_churn(customers, seg)
        churn_source = f"empirical, {events} churn events over {exposure:.0f} customer-months observed"
    ltv = lib.compute_ltv(acv, churn)
    seg_cac = next(r["cac"] for r in cac_rows if r["segment"] == seg)
    payback = lib.compute_payback_months(seg_cac, acv)
    ltv_rows.append({
        "segment": seg, "acv": round(acv, 0), "annual_churn": round(churn, 4),
        "churn_source": churn_source,
        "ltv_capped_3yr": round(ltv["ltv_capped"], 0), "ltv_naive": round(ltv["ltv_naive"], 0),
        "ltv_cac_capped": round(ltv["ltv_capped"] / seg_cac, 2) if seg_cac == seg_cac else None,
        "ltv_cac_naive": round(ltv["ltv_naive"] / seg_cac, 2) if seg_cac == seg_cac else None,
        "payback_months_trailing_cac": round(payback, 1),
    })
ltv_df = pd.DataFrame(ltv_rows)
ltv_df

,segment,acv,annual_churn,churn_source,ltv_capped_3yr,ltv_naive,ltv_cac_capped,ltv_cac_naive,payback_months_trailing_cac
0,Enterprise,99868.0,0.0800,"benchmark (Optifai), Enterprise retention unme...",233692.0,973716.0,2.12,8.82,17.0
1,Mid-Market,35012.0,0.1184,"empirical, 24 churn events over 2297 customer-...",81928.0,230612.0,4.60,12.94,7.8
2,SMB,11825.0,0.2862,"empirical, 100 churn events over 3610 customer...",27670.0,32231.0,3.23,3.76,11.2


**Why the capped LTV is the headline, not the naive one.** Naive `1/churn` gives Enterprise a 12.5-year implied customer life and an LTV:CAC ratio (shown above) that makes the segment look comfortably healthy by the standard 3:1 rule of thumb. No five-year-old company (Anchorpoint, founded 2021) should book a 12.5-year customer life; the naive method overstates LTV for any low-churn segment. The capped method (3-year ceiling) is reported as the primary figure for exactly this reason, and the naive figure is retained above only as a labelled contrast, per the locked rigor requirement in Question 1.

### Enterprise LTV sensitivity to the benchmark churn assumption

Since Enterprise churn is a benchmark input, not a measured output, the LTV figure above is shown across the benchmark's stated sensitivity range (5-10%, Optifai) rather than as a single number treated as precise.

In [4]:
acv_ent = deals[(deals["segment"] == "Enterprise") & (deals["deal_won"])]["deal_size_usd"].mean()
sens_rows = []
for churn in [0.05, 0.08, 0.10]:
    ltv = lib.compute_ltv(acv_ent, churn)
    sens_rows.append({"assumed_annual_churn": churn, "ltv_capped_3yr": round(ltv["ltv_capped"], 0)})
pd.DataFrame(sens_rows)

,assumed_annual_churn,ltv_capped_3yr
0,0.05,233692.0
1,0.08,233692.0
2,0.10,233692.0


## Question 2 (unit-economics half): segment divergence, pre-push vs. post-push

Same-period era CAC (no lag shift, see methodology note above). Enterprise pre-push era: 2024-07-01 to 2025-01-01 (6.0 months, founder-led). Post-push era: 2025-01-01 to 2026-08-31 (19.9 months, new AE team).

In [5]:
era_rows = []
for seg in lib.SEGMENTS:
    pre = lib.compute_era_cac(deals, spend, seg, lib.WINDOW_START, lib.ENTERPRISE_PUSH_DATE, "pre-push")
    post = lib.compute_era_cac(deals, spend, seg, lib.ENTERPRISE_PUSH_DATE, lib.WINDOW_END, "post-push")
    acv_pre = deals[(deals["segment"] == seg) & (deals["deal_won"]) & (deals["close_date"] < lib.ENTERPRISE_PUSH_DATE)]["deal_size_usd"].mean()
    acv_post = deals[(deals["segment"] == seg) & (deals["deal_won"]) & (deals["close_date"] >= lib.ENTERPRISE_PUSH_DATE)]["deal_size_usd"].mean()
    era_rows.append({
        "segment": seg, "pre_push_cac": round(pre["cac"], 0), "pre_push_wins": pre["wins"],
        "post_push_cac": round(post["cac"], 0), "post_push_wins": post["wins"],
        "cac_multiple": round(post["cac"] / pre["cac"], 2) if pre["cac"] == pre["cac"] and pre["cac"] > 0 else None,
        "pre_push_payback_mo": round(lib.compute_payback_months(pre["cac"], acv_pre), 1),
        "post_push_payback_mo": round(lib.compute_payback_months(post["cac"], acv_post), 1),
    })
era_df = pd.DataFrame(era_rows)
era_df

,segment,pre_push_cac,pre_push_wins,post_push_cac,post_push_wins,cac_multiple,pre_push_payback_mo,post_push_payback_mo
0,Enterprise,11520.0,25,121212.0,44,10.52,1.9,18.1
1,Mid-Market,21951.0,41,18072.0,166,0.82,10.9,7.7
2,SMB,6316.0,95,8197.0,244,1.30,9.0,10.3


**Reading this table.** Enterprise CAC rose roughly 10x between the founder-led era and the new-team era (era-to-era, same-period method), and payback stretched from under two months to a little over a year and a half. Mid-Market and SMB CAC moved much less over the same calendar split, in the same or opposite direction. The pre-push Enterprise figures reflect a minimal, founder-led motion running on almost no dedicated spend (`ENTERPRISE_PRE_PUSH_SPEND_FRACTION` scaled Enterprise spend down before the push), not a scalable benchmark: the founder era was cheap because it was small, not because it was an efficient way to run Enterprise sales at the company's current size. The post-push figure, run against a fully staffed team, is the one that matters for the "should we keep the 8 seats" decision.

This is the blended-CAC form of the same divergence that shows up in win rate: aggregate CAC (trailing 12mo, all segments) looks like a single reasonable number, and the segment decomposition is where the story is. See `03_diagnostics.ipynb` for the win-rate side of this and Question 7 for the budget-share framing.

## Reproducibility note

Re-running `python src/validate_dataset.py --phase 3` at the end of this phase confirms `data/raw/` is untouched by this notebook. No writes to `data/raw/` occur anywhere in this analysis.

## Shot list chart

Saved to `notebooks/figures/` for embedding in the Phase 4 case study. See `docs/screenshot_shot_list.md`.

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
os.makedirs("figures", exist_ok=True)

# Brand palette for the chart restyle, locked 2026-08-21 (docs/decisions.md, Decision 24).
# Derived from the live site guide (05_EA_Neyda_AI/brand-guidelines/brand-style-guide.md),
# not the healthcare-ops-finance palette in 04_Brand_and_Portfolio/brand/, which is scoped
# to that project only.
BG = "#FAF8F5"       # site Background
GRID = "#E6E2DC"      # site Line
INK = "#1A1A1A"       # site Ink, titles
MUTED = "#5C5C5C"     # site Muted, axis labels, captions, legend, neutral category
SERIES_A = "#291752"  # site Accent Dark, baseline series (pre-push, blended, close-date)
SERIES_B = "#B5502E"  # new terracotta, contrast series (post-push, Enterprise, open-date)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED,
    "axes.titlecolor": INK,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 10.5,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
    "font.family": "Inter",
})


In [7]:
# Shot 3: CAC and payback by segment, era comparison (pre-push vs post-push)
segs = list(era_df["segment"])
x = list(range(len(segs)))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar([i - width / 2 for i in x], era_df["pre_push_cac"], width, label="Pre-push", color=SERIES_A)
axes[0].bar([i + width / 2 for i in x], era_df["post_push_cac"], width, label="Post-push", color=SERIES_B)
axes[0].set_xticks(x); axes[0].set_xticklabels(segs)
axes[0].set_ylabel("CAC (USD)")
axes[0].set_title("CAC by segment, era comparison")
axes[0].legend()

axes[1].bar([i - width / 2 for i in x], era_df["pre_push_payback_mo"], width, label="Pre-push", color=SERIES_A)
axes[1].bar([i + width / 2 for i in x], era_df["post_push_payback_mo"], width, label="Post-push", color=SERIES_B)
axes[1].set_xticks(x); axes[1].set_xticklabels(segs)
axes[1].set_ylabel("Payback (months)")
axes[1].set_title("Payback by segment, era comparison")
axes[1].legend()

plt.tight_layout()
plt.savefig("figures/03_cac_payback_by_segment_period.png", dpi=150)
plt.show()
print("saved figures/03_cac_payback_by_segment_period.png")


saved figures/03_cac_payback_by_segment_period.png
